# آموزش PPO برای کنترل ارتفاع کوادکوپتر — ۵۰٬۰۰۰ گام

این نوت‌بوک نسخهٔ اصلاح‌شدهٔ آموزش را اجرا می‌کند. مهم‌ترین اصلاح این است که هر چهار اکشن PPO در بازهٔ متقارن ‎[-1, 1]‎ هستند و گاز به‌صورت جداگانه به بازهٔ عملیاتی تبدیل می‌شود؛ بنابراین خروجی گاز به‌خاطر بریده‌شدن توزیع PPO در صفر قفل نمی‌شود. گاز در بازهٔ محدودتر ۰٫۲۰ تا ۰٫۵۰ نگه داشته شده و تابع پاداش برای نزدیک‌ماندن به ارتفاع هدف امتیاز محدود و قابل‌تفسیر می‌دهد؛ گزارش ارزیابی هم نرخ سقوط و سهم زمان هاور پایدار را ثبت می‌کند.

**محدودیت:** این فقط آموزش در شبیه‌ساز است. تنظیمات فیزیکی هنوز جایگزین اندازه‌گیری واقعی نیستند و مدل برای پرواز واقعی تأیید نشده است.

In [ ]:
!git clone https://github.com/samanramezani1377-hub/3d-fly-test.git
%cd /content/3d-fly-test
!python -m pip install -r training/requirements.txt


## ساخت هستهٔ فیزیک C++

اگر این مرحله خطا داد، آموزش را اجرا نکن و متن خطا را بفرست.

In [ ]:
!python training/build_native_core.py


## اجرای آموزش اصلی

آموزش روی **۵۰٬۰۰۰ گام** تنظیم شده است. هر ۵٬۰۰۰ گام checkpoint ذخیره و ارزیابی انجام می‌شود؛ بهترین مدل ارزیابی‌شده هم جداگانه نگه‌داری می‌شود.

In [ ]:
TIMESTEPS = 50_000
RUN_NAME = 'ppo-hover-50000'
!python training/train_ppo.py --timesteps {TIMESTEPS} --seed 42 --run-name {RUN_NAME} --save-freq 5000 --eval-freq 5000 --eval-episodes 10


## ارزیابی مدل نهایی

این ارزیابی بعد از پایان آموزش اجرا می‌شود و نتیجه را داخل پوشهٔ اجرا ذخیره می‌کند.

In [ ]:
!python training/evaluate.py --model training/runs/{RUN_NAME}/final_model.zip --episodes 10 --output training/runs/{RUN_NAME}/final-evaluation.json
!if [ -f training/runs/{RUN_NAME}/best/best_model.zip ]; then python training/evaluate.py --model training/runs/{RUN_NAME}/best/best_model.zip --episodes 10 --output training/runs/{RUN_NAME}/best-evaluation.json; fi


## دانلود همهٔ خروجی‌ها

ZIP شامل مدل نهایی، بهترین مدل ارزیابی‌شده، checkpointها، لاگ‌ها و گزارش‌های ارزیابی است. آن را قبل از بستن Colab دانلود کن.

In [ ]:
!cd training/runs && zip -qr /content/{RUN_NAME}-results.zip {RUN_NAME}
from google.colab import files
files.download(f'/content/{RUN_NAME}-results.zip')


## نکتهٔ مهم برای ادامهٔ کار

۵۰٬۰۰۰ گام شروع بهتری از اجرای ۵٬۰۰۰ گامی است، اما تضمین نمی‌کند که کنترلر حتماً یاد گرفته باشد. معیار تصمیم، گزارش ارزیابی، نرخ برخورد، نرخ موفقیت هاور، سهم زمان پایدار، خطای ارتفاع و سرعت عمودی است. اگر نتیجه خوب نبود، گزارش ZIP را در همین گفتگو بفرست تا بر اساس شواهد مرحلهٔ بعد را اصلاح کنیم. این مدل را روی کوادکوپتر واقعی اجرا نکن.